In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np 
from tqdm import tqdm
import stackview as sv
from functools import partial

from codex.data.codex_dataset import CodexDataset
from codex.preprocessing.background_correction import LinearInterpolationCorrector, AutofluorescenceCorrector
from codex.preprocessing.nodes import BackgroundCorrectionNode

In [ ]:
out_dir = Path("../../outputs/")
out_dir.mkdir(parents=True, exist_ok=True)

ds = CodexDataset(
    root_dir=Path("../../outputs/stitching"),
    mode="raw",
    lazy_loading=True,
    read_markers=False,
)

node = BackgroundCorrectionNode(
    # algorithm=partial(LinearInterpolationCorrector, nuclear_channel=1, skip_markers=["Blank", "Empty"], first_cycle=1),
    algorithm=partial(AutofluorescenceCorrector, skip_markers=["Blank", "Empty"]),
    ds=ds,
    out_dir=out_dir,
    n_jobs=4,
)

node.run()

In [ ]:
ds = CodexDataset(
    root_dir=Path("../../outputs/stitching"),
    mode="raw",
    lazy_loading=True,
    read_markers=False,
)
ds.sort_by(["cycle", "channel"])
ds.group_channels()
ds.df

In [ ]:
ds2 = CodexDataset(
    root_dir=Path("../../outputs/background_correction"),
    mode="raw",
    lazy_loading=True,
    read_markers=False,
)
ds2.sort_by(["cycle", "channel"])
ds2.group_channels()
ds2.df

In [ ]:
from codex.utils.img_utils import resize


img1 = ds[2]["img"]
img2 = ds2[2]["img"]

# factor = 0.07
sv.curtain(resize(img1[2], 0.05), resize(img2[2], 0.05), colormap="turbo", curtain_colormap="turbo", continuous_update=True)